# บทที่ 12 - การลดประวัติการแชทด้วย Agent Scratchpad

สมุดบันทึกนี้แสดงให้เห็นวิธีการจัดการบริบทในบทสนทนาแบบยาวโดยใช้ Microsoft Agent Framework เมื่อบทสนทนาเพิ่มขึ้น จำนวนโทเค็นก็จะเพิ่มขึ้น — สุดท้ายเกินหน้าต่างบริบทของโมเดล เราจัดการปัญหานี้ด้วย **รูปแบบการสรุปบริบท** และ **agent scratchpad** สำหรับหน่วยความจำถาวร

## สิ่งที่คุณจะได้เรียนรู้:
1. **ทำไมการจัดการบริบทจึงสำคัญ**: เข้าใจขีดจำกัดโทเค็นและหน้าต่างบริบท
2. **Agent ที่ตระหนักบริบท**: การสร้าง agent ที่จัดการบริบทของบทสนทนาเอง
3. **รูปแบบการสรุปบริบท**: ใช้เครื่องมือเพื่อย่อประวัติการสนทนา
4. **Agent Scratchpad**: หน่วยความจำถาวรที่ยังคงอยู่แม้ลดบริบท

## ความต้องการเบื้องต้น:
- ตั้งค่า Azure OpenAI พร้อมตัวแปรสภาพแวดล้อม
- ความเข้าใจพื้นฐานเกี่ยวกับแนวคิด agent จากบทเรียนก่อนหน้า


## การตั้งค่า


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## ทำไมการจัดการบริบทจึงสำคัญ

ทุก LLM มีขนาด **หน้าต่างบริบท** ที่จำกัด — จำนวนโทเค็นสูงสุดที่สามารถประมวลผลได้ในคำขอเดียว เมื่อการสนทนาแบบหลายรอบดำเนินไป:

- **จำนวนโทเค็นเพิ่มขึ้นเชิงเส้น** กับแต่ละข้อความของผู้ใช้และการตอบกลับของผู้ช่วย
- **โทเค็นคำสั่งมีผลต่อค่าใช้จ่ายสูงสุด** เพราะประวัติทั้งหมดถูกส่งซ้ำในแต่ละรอบ
- ในที่สุดสนทนาจะ **เกินขนาดหน้าต่างบริบท** และโมเดลจะตัดหรือเกิดข้อผิดพลาด

### กลยุทธ์ในการจัดการบริบท

| กลยุทธ์ | วิธีการทำงาน | ข้อแลกเปลี่ยน |
|---|---|---|
| **การตัดทอน** | ตัดข้อความที่เก่าที่สุด | สูญเสียบริบทตอนต้น |
| **การสรุป** | รวบรวมข้อความเก่าเป็นสรุป | รายละเอียดบางส่วนหายไป แต่ประเด็นสำคัญยังอยู่ |
| **บันทึกช่วยจำ / หน่วยความจำภายนอก** | เก็บข้อเท็จจริงสำคัญนอกการสนทนา | ต้องเรียกใช้เครื่องมือ แต่ยังคงอยู่แม้ลดข้อความ |

ในโน้ตบุ๊กนี้เราจะผสมผสาน **การสรุป** กับ **เครื่องมือบันทึกช่วยจำ** เพื่อให้นายหน้าสามารถรักษาความต่อเนื่องได้แม้ประวัติการสนทนาจะถูกย่อให้สั้นลง


## การสร้างเอเจนต์ที่มีความตระหนักถึงบริบท


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## การจำลองการสนทนายาว

มาลองเดินผ่านการสนทนาหลายรอบเพื่อดูว่าบริบทสะสมอย่างไร ตัวแทนควรจดจำรายละเอียดสำคัญ (ความชอบ งบประมาณ วันที่เดินทาง) ข้ามรอบและแสดงความต่อเนื่องออกมา


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

สังเกตว่าตัวแทนยังคงบริบทจากรอบก่อนหน้า — มันรู้เกี่ยวกับญี่ปุ่น ซูชิ วัด การถ่ายภาพ งบประมาณ $3000 การเดินทางคนเดียว และช่วงเวลาประมาณเดือนเมษายน ในบทสนทนาสั้น ๆ นี้ทำงานได้ดี แต่เมื่อบทสนทนาเพิ่มขึ้น ประวัติทั้งหมดจะมีค่าใช้จ่ายสูงในการส่งซ้ำ

เรามาต่อบทสนทนากันด้วยรอบเพิ่มเติมเพื่อดูการสะสมบริบท:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## รูปแบบการสรุปบริบท

เมื่อการสนทนาเติบโตขึ้น เราสามารถใช้ **เครื่องมือสรุป** เพื่อย่อบริบทที่สะสมมาให้กลายเป็นรูปแบบที่กระชับ ตัวแทนจะเรียกใช้เครื่องมือนี้เพื่อบันทึกรายละเอียดสำคัญ เพื่อให้แม้ว่าข้อความเก่าจะถูกลบไป ข้อมูลสำคัญก็ยังคงถูกเก็บรักษาไว้

รูปแบบนี้เป็นองค์ประกอบพื้นฐานสำหรับการลดประวัติที่ซับซ้อนมากขึ้น:
1. ตัวแทนจะระบุข้อเท็จจริงสำคัญจากการสนทนา
2. เรียกใช้เครื่องมือสรุปเพื่อเก็บรักษาข้อเท็จจริงเหล่านั้น
3. สามารถลบข้อความเก่าได้อย่างปลอดภัย เพราะสรุปนั้นจับภาพสิ่งที่สำคัญไว้แล้ว

ด้านล่างนี้เรากำหนดเครื่องมือ `summarize_preferences` ซึ่งตัวแทนสามารถเรียกใช้เพื่อลงบันทึกสรุปสั้น ๆ ของสิ่งที่ตัวเองได้เรียนรู้


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้วิธีการจัดการบริบทในการสนทนาแบบเอเจนต์ที่ทำงานเป็นเวลานานโดยใช้ Microsoft Agent Framework:

### แนวคิดสำคัญ
- **หน้าต่างบริบทมีขอบเขตจำกัด** — ทุกโทเค็นในประวัติการสนทนาจะมีค่าใช้จ่ายและถูกนับรวมในขีดจำกัด
- **เครื่องมือสรุปข้อความ** ช่วยให้เอเจนต์บีบอัดบริบทที่สะสมไว้ให้กลายเป็นสรุปที่กะทัดรัด เพื่อลดการใช้โทเค็นในขณะที่ยังคงรักษาข้อมูลสำคัญไว้
- **สมุดร่างของเอเจนต์** มีหน่วยความจำภายนอกที่ถาวรซึ่งคงอยู่แม้จะมีการลดขนาดการสนทนาใดๆ

### สิ่งที่คุณสร้างขึ้น
- **เอเจนต์ที่รับรู้บริบท** ที่รักษาความต่อเนื่องในระหว่างการสนทนาแบบหลายรอบ
- **เครื่องมือสรุปข้อความ** (`summarize_preferences`) ที่บันทึกรายละเอียดสำคัญของผู้ใช้ในรูปแบบกะทัดรัด
- **การสนทนาแบบหลายรอบ** ที่แสดงให้เห็นการเก็บรักษาบริบทและการจัดการการเปลี่ยนแปลง

### การใช้งานในโลกจริง
- **บอทบริการลูกค้า**: จำความชอบขณะให้การสนับสนุนระยะยาว
- **ผู้ช่วยส่วนตัว**: ติดตามโครงการที่กำลังดำเนินอยู่อย่างไม่ต้องอธิบายบริบทซ้ำ
- **ติวเตอร์การศึกษา**: รักษาความคืบหน้าของนักเรียนผ่านหลายการติดต่อ

### ขั้นตอนถัดไป
- นำเครื่องมือสมุดร่างที่สมบูรณ์มาทำงานร่วมกับการเก็บถาวรแบบไฟล์
- เพิ่มการตัดประวัติอัตโนมัติหลังการสรุปข้อความ
- รวมเข้ากับฐานข้อมูลเวกเตอร์เพื่อการค้นหาหน่วยความจำเชิงความหมาย
- สร้างเอเจนต์ที่สามารถกลับมาสานต่อการสนทนาหลังจากหลายวันโดยมีบริบทครบถ้วน


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือตำหนิได้ เอกสารต้นฉบับในภาษาดั้งเดิมถือเป็นแหล่งข้อมูลที่เชื่อถือได้ ในกรณีข้อมูลที่มีความสำคัญ ขอแนะนำให้ใช้บริการแปลโดยผู้เชี่ยวชาญมนุษย์ เราจะไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความผิดใด ๆ ที่เกิดจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
